# Reconciling Schema Drift

In **Notebook 03**, we combined four termly snapshots into single `schools_combined` and `pupils_combined` tables in silver. The dynamic combination preserved **every column from every snapshot** — which means columns that were renamed between terms now appear as separate fields:

| Concept | Autumn 2024 / 2025 | Spring 2025 / Summer 2025 |
| --- | --- | --- |
| School identifier | `school_urn` | `urn` |
| School name | `school_name` | `establishment_name` |
| Location | `city` | `local_authority` |
| School type | `school_type` | `establishment_type` |
| Phase | `phase` | `phase_of_education` |
| Pupil first name | `first_name` | `forename` |
| Pupil last name | `last_name` | `surname` |
| Gender | `gender` | `sex` |
| Date of birth | `date_of_birth` | `dob` |

Rows from one set of snapshots have values in one column and `NULL` in the other. This makes every downstream query harder — cardinality checks, joins, and deduplication all need `COALESCE` wrappers.

This notebook fixes these problems before they appear by **unifying the split columns** into new `schools_reconciled` and `pupils_reconciled` tables in silver. The original `_combined` tables are preserved so every step in the pipeline remains auditable.

> **Note:** All data in this module is **entirely synthetic** and does not represent any real schools, pupils, or individuals.

## Schools

Let's start by previewing the combined schools table to see the split columns in practice.

In [0]:
USE CATALOG catalog_40_copper_analyst_training;

In [0]:
-- Preview the combined schools table — notice the split columns
-- Autumn rows have school_urn/school_name/city; spring/summer rows have urn/establishment_name/local_authority
SELECT
  school_urn
  ,urn
  ,school_name
  ,establishment_name
  ,city
  ,local_authority
  ,school_type
  ,establishment_type
  ,phase
  ,phase_of_education
  ,source_table
FROM silver.schools_combined
ORDER BY COALESCE(school_urn, urn), source_table
LIMIT 12;

Each row has values in **one** set of columns and `NULL` in the other. For example, an autumn row has a `school_urn` but `NULL` for `urn`; a spring row has the reverse.

We fix this with `COALESCE(column_a, column_b)` — which returns the first non-null value, unifying the split columns into a single field per concept.

We'll also extract the **term** and **year** from `source_table` into their own columns. Currently `source_table` holds values like `schools_autumn_2024` — encoding both the term (`autumn`) and the academic year (`2024`) in a single string. These are separate real-world dimensions that form part of the natural key, so they deserve their own fields.

In [0]:
-- Reconcile the split columns into a single clean schema
-- COALESCE picks the non-null value from whichever column has it
-- SPLIT extracts term and year from source_table (e.g. 'schools_autumn_2024' → 'autumn', '2024')
CREATE OR REPLACE TEMPORARY VIEW silver_schools_reconciled AS
SELECT
  COALESCE(school_urn, urn) AS school_urn
  ,COALESCE(school_name, establishment_name) AS school_name
  ,COALESCE(city, local_authority) AS location
  ,COALESCE(school_type, establishment_type) AS school_type
  ,status
  ,COALESCE(phase, phase_of_education) AS phase
  ,region
  ,metadata_json
  ,source_table
  ,SPLIT(source_table, '_')[1] AS term
  ,CAST(SPLIT(source_table, '_')[2] AS INT) AS year
FROM silver.schools_combined;

The reconciled table now has a single column per concept — no more split fields. The `term` and `year` columns have been extracted from `source_table`, and `source_table` itself is retained for full traceability back to the original bronze snapshot.

Let's verify that the key fields are fully populated — any remaining `NULL`s would indicate a row that had neither variant of the column populated in the source data, which is a genuine data quality issue rather than schema drift.

In [0]:
-- Verify: no more split columns, no more NULLs in school_urn
SELECT
  COUNT(*) AS total_rows
  ,COUNT(school_urn) AS non_null_school_urn
  ,COUNT(school_name) AS non_null_school_name
  ,COUNT(location) AS non_null_location
FROM silver_schools_reconciled;

## Pupils

The pupils table has the same issue — `first_name`/`forename`, `last_name`/`surname`, `gender`/`sex`, `date_of_birth`/`dob`, and `school_urn`/`urn` are all split across columns. Let's preview and fix.

In [0]:
-- Preview the combined pupils table — notice the split columns
SELECT
  pupil_id
  ,first_name
  ,forename
  ,last_name
  ,surname
  ,gender
  ,sex
  ,date_of_birth
  ,dob
  ,school_urn
  ,urn
  ,source_table
FROM silver.pupils_combined
ORDER BY pupil_id, source_table
LIMIT 12;

In [0]:
-- Reconcile the split columns into a single clean schema
-- Same COALESCE + SPLIT pattern as schools
CREATE OR REPLACE TEMPORARY VIEW silver_pupils_reconciled AS
SELECT
  pupil_id
  ,COALESCE(first_name, forename) AS first_name
  ,COALESCE(last_name, surname) AS last_name
  ,COALESCE(gender, sex) AS gender
  ,COALESCE(date_of_birth, dob) AS date_of_birth
  ,COALESCE(school_urn, urn) AS school_urn
  ,year_group
  ,ethnicity
  ,metadata_json
  ,source_table
  ,SPLIT(source_table, '_')[1] AS term
  ,CAST(SPLIT(source_table, '_')[2] AS INT) AS year
FROM silver.pupils_combined;

In [0]:
-- Verify: no more split columns, no more NULLs in key fields
SELECT
  COUNT(*) AS total_rows
  ,COUNT(pupil_id) AS non_null_pupil_id
  ,COUNT(first_name) AS non_null_first_name
  ,COUNT(school_urn) AS non_null_school_urn
FROM silver_pupils_reconciled;

## Summary

Both reconciled tables now have **clean, unified schemas**:

| Before (`_combined`) | After (`_reconciled`) |
| --- | --- |
| `school_urn` + `urn` (one always NULL) | `school_urn` (always populated) |
| `school_name` + `establishment_name` | `school_name` |
| `city` + `local_authority` | `location` |
| `school_type` + `establishment_type` | `school_type` |
| `phase` + `phase_of_education` | `phase` |
| `first_name` + `forename` | `first_name` |
| `last_name` + `surname` | `last_name` |
| `gender` + `sex` | `gender` |
| `date_of_birth` + `dob` | `date_of_birth` |
| `source_table` (e.g. `schools_autumn_2024`) | `source_table` retained, plus new `term` and `year` columns |

The new `term` and `year` columns make the **natural key** explicit — `school_urn` + `term` + `year` for schools, `pupil_id` + `term` + `year` for pupils — without needing to parse `source_table` in every downstream query.

The original `_combined` tables are preserved in silver, so you can always trace back to the raw combined data if needed.

Downstream notebooks read from the `_reconciled` tables in the `silver` schema which contain the same data as `silver_schools_reconciled` and `silver_pupils_reconciled` in this notebook:
* `catalog_40_copper_analyst_training.silver.schools_reconciled`
* `catalog_40_copper_analyst_training.silver.pupils_reconciled`

### What's next

* **Notebook 05 — Understanding Your Data** will check cardinality, keys, and referential integrity on these clean tables